In [1]:
import sys
from pathlib import Path

print(sys.executable)
print(Path.cwd())

c:\temp\python_learning\ml_projects\ml_projects_batch_01\02_telco_customer_churn\.venv\Scripts\python.exe
c:\temp\python_learning\ml_projects\ml_projects_batch_01\02_telco_customer_churn\notebooks


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [3]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)

df.shape

(7043, 21)

In [4]:
target_col = "Churn"

y = df[target_col].map({
    "No": 0,
    "Yes": 1,
})

y.value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [5]:
id_cols = ["customerID"]

X = df.drop(columns=id_cols + [target_col])

X.shape

(7043, 19)

In [6]:
X["TotalCharges"] = pd.to_numeric(X["TotalCharges"], errors="coerce")

In [7]:
numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]

categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]

all_features = numeric_features + categorical_features

len(numeric_features), len(categorical_features), len(all_features), X.shape[1]

(4, 15, 19, 19)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [9]:
split_distribution = pd.DataFrame({
    "full": y.value_counts(normalize=True).sort_index(),
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
})

split_distribution

,full,train,test
Churn,,,
0,0.73463,0.734647,0.734564
1,0.26537,0.265353,0.265436


In [10]:
split_counts = pd.DataFrame({
    "full": y.value_counts().sort_index(),
    "train": y_train.value_counts().sort_index(),
    "test": y_test.value_counts().sort_index(),
})

split_counts

,full,train,test
Churn,,,
0,5174,4139,1035
1,1869,1495,374


## Stage 4 setup notes

### Goal

Controlled comparison of stronger model families against the current baseline.

Current strongest simple baseline from Stage 3:

- `LogisticRegression(class_weight="balanced")`

### Task framing

- Task type: binary classification.
- Target: `Churn`.
- Positive class: `Yes`, encoded as `1`.
- Negative class: `No`, encoded as `0`.

### Data preparation

- Raw dataset is loaded from `data/raw`.
- `customerID` is excluded from `X`.
- `Churn` is excluded from `X`.
- `TotalCharges` is converted to numeric with `errors="coerce"`.
- The same split logic from Stage 3 is reused:
  - `test_size=0.2`
  - `random_state=42`
  - `stratify=y`

### Leakage control

- `X_test` and `y_test` are created but must stay untouched.
- Model comparison must use only `X_train` and `y_train`.
- No model is evaluated on `X_test`.
- No hyperparameter tuning is performed.
- No threshold tuning is performed.
- Preprocessing must be placed inside sklearn `Pipeline` / `ColumnTransformer`.
- Imputers, scalers and encoders must be fitted only inside CV folds.

Ячейка 11 — imports for preprocessing, models, CV

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)

from sklearn.model_selection import StratifiedKFold, cross_validate

Ячейка 12 — preprocessing

In [12]:
numeric_preprocessor_scaled = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

numeric_preprocessor_unscaled = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor_scaled, numeric_features),
        ("cat", categorical_preprocessor, categorical_features),
    ]
)

preprocessor_unscaled = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor_unscaled, numeric_features),
        ("cat", categorical_preprocessor, categorical_features),
    ]
)

Ячейка 13 — models

In [13]:
model_specs = {
    "Logistic Regression": {
        "preprocessor": preprocessor_scaled,
        "model": LogisticRegression(max_iter=1000, random_state=42),
    },
    "Logistic Regression balanced": {
        "preprocessor": preprocessor_scaled,
        "model": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
    },
    "Decision Tree max_depth=3": {
        "preprocessor": preprocessor_unscaled,
        "model": DecisionTreeClassifier(
            max_depth=3,
            random_state=42,
        ),
    },
    "Random Forest": {
        "preprocessor": preprocessor_unscaled,
        "model": RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1,
        ),
    },
    "Extra Trees": {
        "preprocessor": preprocessor_unscaled,
        "model": ExtraTreesClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1,
        ),
    },
    "Gradient Boosting": {
        "preprocessor": preprocessor_unscaled,
        "model": GradientBoostingClassifier(
            random_state=42,
        ),
    },
    "Hist Gradient Boosting": {
        "preprocessor": preprocessor_unscaled,
        "model": HistGradientBoostingClassifier(
            random_state=42,
        ),
    },
}

Ячейка 14 — CV and metrics

In [14]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

Ячейка 15 — cross-validation loop

In [17]:
cv_results = []

for model_name, spec in model_specs.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", spec["preprocessor"]),
            ("model", spec["model"]),
        ]
    )
    
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )
    
    result = {"model": model_name}
    
    for metric_name in scoring.keys():
        result[f"{metric_name}_mean"] = scores[f"test_{metric_name}"].mean()
        result[f"{metric_name}_std"] = scores[f"test_{metric_name}"].std()
    
    cv_results.append(result)

results_df = pd.DataFrame(cv_results)

results_df

,model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,Logistic Regression,0.802096,0.011584,0.652899,0.025457,0.543144,0.041103,0.592316,0.029612,0.846137,0.012545,0.661430,0.019469
1,Logistic Regression balanced,0.748138,0.015307,0.516482,0.018901,0.801338,0.037933,0.627927,0.023225,0.845950,0.012379,0.660023,0.019536
2,Decision Tree max_depth=3,0.789494,0.004916,0.699093,0.020446,0.363880,0.025593,0.477956,0.021607,0.816043,0.006982,0.563056,0.015005
3,Random Forest,0.785058,0.012944,0.623018,0.034504,0.483612,0.024540,0.544243,0.025604,0.820395,0.012495,0.613284,0.029468
4,Extra Trees,0.768373,0.010987,0.577774,0.024678,0.470234,0.027547,0.518431,0.026469,0.786460,0.015900,0.552253,0.036585
5,Gradient Boosting,0.803340,0.011003,0.660566,0.028851,0.533779,0.023421,0.590200,0.022660,0.847741,0.012235,0.667707,0.023653
6,Hist Gradient Boosting,0.797483,0.012861,0.649110,0.032560,0.516388,0.029126,0.574919,0.028141,0.837821,0.007467,0.649527,0.021120


Ячейка 16 — comparison table sorted by F1

In [18]:
comparison_cols = [
    "model",
    "accuracy_mean",
    "precision_mean",
    "recall_mean",
    "f1_mean",
    "roc_auc_mean",
    "pr_auc_mean",
]

model_comparison = (
    results_df[comparison_cols]
    .sort_values("f1_mean", ascending=False)
    .reset_index(drop=True)
)

model_comparison_rounded = model_comparison.copy()

metric_cols = [
    "accuracy_mean",
    "precision_mean",
    "recall_mean",
    "f1_mean",
    "roc_auc_mean",
    "pr_auc_mean",
]

model_comparison_rounded[metric_cols] = model_comparison_rounded[metric_cols].round(4)

model_comparison_rounded

,model,accuracy_mean,precision_mean,recall_mean,f1_mean,roc_auc_mean,pr_auc_mean
0,Logistic Regression balanced,0.7481,0.5165,0.8013,0.6279,0.8459,0.6600
1,Logistic Regression,0.8021,0.6529,0.5431,0.5923,0.8461,0.6614
2,Gradient Boosting,0.8033,0.6606,0.5338,0.5902,0.8477,0.6677
3,Hist Gradient Boosting,0.7975,0.6491,0.5164,0.5749,0.8378,0.6495
4,Random Forest,0.7851,0.6230,0.4836,0.5442,0.8204,0.6133
5,Extra Trees,0.7684,0.5778,0.4702,0.5184,0.7865,0.5523
6,Decision Tree max_depth=3,0.7895,0.6991,0.3639,0.4780,0.8160,0.5631


Ячейка 17 — best models by key metrics

In [19]:
best_by_f1 = model_comparison_rounded.sort_values("f1_mean", ascending=False).head(3)
best_by_recall = model_comparison_rounded.sort_values("recall_mean", ascending=False).head(3)
best_by_pr_auc = model_comparison_rounded.sort_values("pr_auc_mean", ascending=False).head(3)

best_by_f1

,model,accuracy_mean,precision_mean,recall_mean,f1_mean,roc_auc_mean,pr_auc_mean
0,Logistic Regression balanced,0.7481,0.5165,0.8013,0.6279,0.8459,0.6600
1,Logistic Regression,0.8021,0.6529,0.5431,0.5923,0.8461,0.6614
2,Gradient Boosting,0.8033,0.6606,0.5338,0.5902,0.8477,0.6677


In [20]:
best_by_recall

,model,accuracy_mean,precision_mean,recall_mean,f1_mean,roc_auc_mean,pr_auc_mean
0,Logistic Regression balanced,0.7481,0.5165,0.8013,0.6279,0.8459,0.6600
1,Logistic Regression,0.8021,0.6529,0.5431,0.5923,0.8461,0.6614
2,Gradient Boosting,0.8033,0.6606,0.5338,0.5902,0.8477,0.6677


In [21]:
best_by_pr_auc

,model,accuracy_mean,precision_mean,recall_mean,f1_mean,roc_auc_mean,pr_auc_mean
2,Gradient Boosting,0.8033,0.6606,0.5338,0.5902,0.8477,0.6677
1,Logistic Regression,0.8021,0.6529,0.5431,0.5923,0.8461,0.6614
0,Logistic Regression balanced,0.7481,0.5165,0.8013,0.6279,0.8459,0.6600


Ячейка 18 — compare against balanced Logistic Regression

In [22]:
baseline_name = "Logistic Regression balanced"

baseline_row = model_comparison_rounded[
    model_comparison_rounded["model"] == baseline_name
]

baseline_row

,model,accuracy_mean,precision_mean,recall_mean,f1_mean,roc_auc_mean,pr_auc_mean
0,Logistic Regression balanced,0.7481,0.5165,0.8013,0.6279,0.8459,0.66


## Stage 4 model comparison conclusions

### Evaluation setup

- All models were evaluated only on `X_train` / `y_train`.
- `X_test` / `y_test` were not used.
- Evaluation method: 5-fold `StratifiedKFold` cross-validation.
- Preprocessing was placed inside sklearn `Pipeline`.
- Numeric preprocessing:
  - `SimpleImputer(strategy="median")`
  - `StandardScaler()` for scale-sensitive models
  - no scaler for tree-based models
- Categorical preprocessing:
  - `SimpleImputer(strategy="most_frequent")`
  - `OneHotEncoder(handle_unknown="ignore")`
- No hyperparameter tuning was performed.
- No threshold tuning was performed.

### Models compared

- `LogisticRegression`
- `LogisticRegression(class_weight="balanced")`
- `DecisionTreeClassifier(max_depth=3)`
- `RandomForestClassifier`
- `ExtraTreesClassifier`
- `GradientBoostingClassifier`
- `HistGradientBoostingClassifier`

### Main results

- Best model by F1:
  - `LogisticRegression(class_weight="balanced")`
  - F1 ≈ 0.6279
- Best model by recall:
  - `LogisticRegression(class_weight="balanced")`
  - recall ≈ 0.8013
- Best model by PR-AUC:
  - `GradientBoostingClassifier`
  - PR-AUC ≈ 0.6677
- Best model by ROC-AUC:
  - `GradientBoostingClassifier`
  - ROC-AUC ≈ 0.8477

### Comparison against current baseline

Current strongest baseline from Stage 3:

- `LogisticRegression(class_weight="balanced")`

The stronger model families did not improve F1 or recall over the balanced Logistic Regression baseline at the default classification threshold.

`GradientBoostingClassifier` slightly improved PR-AUC and ROC-AUC:

- balanced Logistic Regression PR-AUC ≈ 0.6600;
- Gradient Boosting PR-AUC ≈ 0.6677;
- balanced Logistic Regression ROC-AUC ≈ 0.8459;
- Gradient Boosting ROC-AUC ≈ 0.8477.

This suggests that Gradient Boosting may rank churn risk slightly better, but at threshold 0.5 it catches fewer churn cases than balanced Logistic Regression.

### Interpretation

- `LogisticRegression(class_weight="balanced")` remains the best model for churn detection if the main priority is recall and F1 at default threshold.
- Standard `LogisticRegression` and `GradientBoostingClassifier` have better precision and accuracy, but lower recall.
- `RandomForestClassifier` and `ExtraTreesClassifier` did not improve over Logistic Regression baselines in this controlled comparison.
- `DecisionTreeClassifier(max_depth=3)` is interpretable but too weak by recall.

### Recommended candidates for later tuning

Recommended candidates for future stages:

1. `LogisticRegression(class_weight="balanced")`
   - strongest current F1 and recall baseline.
2. `LogisticRegression`
   - nearly identical ROC-AUC and PR-AUC to the balanced version, useful for threshold/class_weight trade-off analysis.
3. `GradientBoostingClassifier`
   - best PR-AUC and ROC-AUC, worth exploring with hyperparameter tuning and threshold tuning later.

### Leakage control

- `X_test` was not evaluated.
- All preprocessing was inside `Pipeline` / `ColumnTransformer`.
- Imputers, scalers, and encoders were fitted only inside CV folds.
- No threshold tuning was performed.
- No hyperparameter tuning was performed.